In [ ]:
!pip install HilbertCurve

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import copy
import tensorflow           as tf
import tensorflow_datasets  as tfds

import numpy                as np
import datetime             as dt
import matplotlib.pyplot    as plt

from hilbertcurve.hilbertcurve import HilbertCurve

from IPython.display import clear_output
from IPython import display

import math
import os
import pickle
import cv2

from google.colab import drive

from keras.utils      import np_utils
import keras
# from keras.datasets   import cifar10, cifar100

from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# dataname = 'cifar10'
# dataname = 'cifar100'
dataname = "imagenette"

# transform = 'Dotprint'
transform = 'Hilbert'

In [ ]:
#############################
## Start of administration ##
#############################
# filename = "cifar10_augmentation2.dat"

startdate = dt.datetime.now()
timestamp = startdate.strftime('%d%m_%H%M%S')

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/AI_Research/Hilbert_AppliedIntelligence/')
os.chdir('Datasets/Augmented')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Fast Hilbert transformation

In [ ]:
def get_dimensions(my_length):
    dimension = 2
    iteration = math.log2(my_length)
    return int(dimension), int(iteration)

def Hilbert_map(dimension,iteration):
    length = int(pow(dimension,iteration))
    set_length = pow(length,dimension)
    my_curve = HilbertCurve(iteration,dimension)

    my_mapping = {}
    for x in range(length):
        for y in range(length):
            my_mapping[x,y] = my_curve.distance_from_point([x,y])

    return my_mapping

def Hilbert_Sequence(mydimension, myiteration):
    my_Hilbert_transformation = Hilbert_map(mydimension,myiteration)
    hilbertmap = list()
    for key in my_Hilbert_transformation:
        hilbertmap.append(my_Hilbert_transformation[key])
    return np.flip(np.transpose(np.array(hilbertmap).reshape(int(mydimension**myiteration),int(mydimension**myiteration))),axis=0)

def Get_Hilbert_Map(sequence, hilbert):
    mappedsequence = list(zip(sequence,hilbert))
    orderedsequence = sorted(mappedsequence,key= lambda x: x[1])
    transformed_image = [i[0] for i in orderedsequence]
    # print("transformed image:", transformed_image[:10])
    return transformed_image
    

def Hilbert_Transformation(images, mydimension, myiteration):
    transformed_images = list()
    hilbert1d = Hilbert_Sequence(mydimension,myiteration).flatten().tolist()
    # print("Hilbert Sequence start: ", hilbert1d[:10])
    image_iterator = 0
    for image in images:
        image = np.array(image)
        transformed_image = list()
        ## clear_output(wait=True)
        ## display.clear_output(wait=True)
        image = image.swapaxes(1,2)
        image = image.swapaxes(0,1)
        ## print("Processing image %i/%i, %.2f pct" %(image_iterator, len(images), 100*image_iterator/len(images)))
        for channel in image:
            channel1d = channel.reshape((channel.shape[0]*channel.shape[1])).tolist()
            transformed_image.append(Get_Hilbert_Map(channel1d, hilbert1d))        
        transformed_image = np.array(transformed_image).swapaxes(0,1)
        transformed_images.append(transformed_image)
        image_iterator += 1

    return np.array(transformed_images,dtype='float32')

def dotprint_2Dto1D_Old(images, mydimension, myiteration):
    transformed_images = list()
    image_iterator = 0
    for image in images:
        image = np.array(image)
        # transformed_image = list()
        ## clear_output(wait=True)
        ## display.clear_output(wait=True)
        # image = image.swapaxes(1,2)
        # image = image.swapaxes(0,1)
        ## print("Processing image %i/%i, %.2f pct" %(image_iterator, len(images), 100*image_iterator/len(images)))
        transformed_image = image.reshape(((int(mydimension)**int(myiteration))**2,3))
        # for channel in image:
        #     channel1d = channel.reshape(((channel.shape[0]*channel.shape[1])**2)).tolist()
        #     transformed_image.append(Get_Hilbert_Map(channel1d, hilbert1d))        
        transformed_image = np.array(transformed_image).swapaxes(0,1)
        transformed_images.append(transformed_image)
        image_iterator += 1
    return np.array(transformed_images,dtype='float32')

def dotprint_2Dto1D(images, mydimension, myiteration):
    transformed_images = list()
    image_iterator = 0
    for image in images:
        image = np.array(image)
        transformed_image = list()
        ## clear_output(wait=True)
        ## display.clear_output(wait=True)
        image = image.swapaxes(1,2)
        image = image.swapaxes(0,1)
        ## print("Processing image %i/%i, %.2f pct" %(image_iterator, len(images), 100*image_iterator/len(images)))
        for channel in image:
            # transformed_image = image.reshape(((int(mydimension)**int(myiteration))**2,3))
            channel1d = channel.flatten() # (((channel.shape[0]*channel.shape[1])**2)).tolist()
            transformed_image.append(channel1d)    
        
        transformed_image = np.array(transformed_image).swapaxes(0,1)
        # print("Transformed image has shape: ", transformed_image.shape)
        transformed_images.append(transformed_image)
        image_iterator += 1
    return np.array(transformed_images,dtype='float32')

In [ ]:
def get_data(dataname, filename):
    #############################
    ## Start Data Processing   ##
    ############################# 
    if dataname == "imagenette":
        class_labels =['tench','English springer','cassette player','chain saw','church','French horn','garbage truck','gas pump','golf ball','parachute']
        nb_classes = len(class_labels)
        ###############################
        ## Augmented set not present ##
        ###############################
        with (open(filename,"rb")) as handle:
            augmented_data = pickle.load(handle)
            handle.close()
        
        x_train = augmented_data["x_train"]
        y_train = augmented_data["y_train"]
        x_test  = augmented_data["x_test"]
        y_test  = augmented_data["y_test"]
        
        input_shape = x_train[0].shape
        my_dimension = 2
        my_iteration = math.log2(x_train.shape[1])

        print(f"x_train shape: {x_train.shape} - y_train shape: {y_train.shape}")
        print(f"x_test shape: {x_test.shape} - y_test shape: {y_test.shape}")
        print(f"data has dimension: {my_dimension} and iteration: {my_iteration}")

    elif dataname == 'cifar10':
        class_labels = ["airplane","car","bird","cat","deer","dog","frog","horse","ship","truck"]
        nb_classes = len(class_labels)
        
        # my_dimension = 2
        # my_iteration = 5

        with (open(filename,"rb")) as handle:
            augmented_data = pickle.load(handle)
            handle.close()
        
        x_train = augmented_data["x_train"]
        y_train = augmented_data["y_train"]
        x_test  = augmented_data["x_test"]
        y_test  = augmented_data["y_test"]
        
        input_shape = x_train[0].shape
        my_dimension = 2
        my_iteration = math.log2(x_train.shape[1])

        print(f"x_train shape: {x_train.shape} - y_train shape: {y_train.shape}")
        print(f"x_test shape: {x_test.shape} - y_test shape: {y_test.shape}")
        print(f"data has dimension: {my_dimension} and iteration: {my_iteration}")

    elif dataname == 'cifar100':
        nb_classes = 100
        # my_dimension = 2
        # my_iteration = 5

        with (open(filename,"rb")) as handle:
            augmented_data = pickle.load(handle)
            handle.close()

        x_train = augmented_data["x_train"]
        y_train = augmented_data["y_train"]
        x_test  = augmented_data["x_test"]
        y_test  = augmented_data["y_test"]
        
        del(augmented_data)
        input_shape = x_train[0].shape
        # input_shape = x_test[0].shape
        my_dimension = 2
        my_iteration = math.log2(x_train.shape[1])
        # my_iteration = math.log2(x_test.shape[1])

        print(f"x_train shape: {x_train.shape} - y_train shape: {y_train.shape}")
        print(f"x_test shape: {x_test.shape} - y_test shape: {y_test.shape}")
        print(f"data has dimension: {my_dimension} and iteration: {my_iteration}")

    return x_train, y_train, x_test, y_test, my_dimension, my_iteration, nb_classes

In [ ]:
if dataname == 'cifar100':
    class_labels = [
    'apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle','bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel','can','castle','caterpillar','cattle','chair','chimpanzee','clock','cloud','cockroach','couch','crab','crocodile','cup',
    'dinosaur','dolphin','elephant','flatfish','forest','fox','girl','hamster','house','kangaroo','computer_keyboard','lamp','lawn_mower','leopard','lion','lizard','lobster','man','maple_tree','motorcycle','mountain','mouse','mushroom',
    'oak_tree','orange','orchid','otter','palm_tree','pear','pickup_truck','pine_tree','plain','plate','poppy','porcupine','possum','rabbit','raccoon','ray','road','rocket','rose','sea','seal','shark','shrew','skunk','skyscraper','snail','snake',
    'spider','squirrel','streetcar','sunflower','sweet_pepper','table','tank','telephone','television','tiger','tractor','train','trout','tulip','turtle','wardrobe','whale','willow_tree','wolf','woman','worm',
]

mapping_coarse_fine = {
    'aquatic mammals': ['beaver', 'dolphin', 'otter', 'seal', 'whale'],
    'fish': ['aquarium_fish', 'flatfish', 'ray', 'shark', 'trout'],
    'flowers': ['orchid', 'poppy', 'rose', 'sunflower', 'tulip'],
    'food containers': ['bottle', 'bowl', 'can', 'cup', 'plate'],
    'fruit and vegetables': ['apple', 'mushroom', 'orange', 'pear',
                             'sweet_pepper'],
    'household electrical device': ['clock', 'computer_keyboard', 'lamp',
                                    'telephone', 'television'],
    'household furniture': ['bed', 'chair', 'couch', 'table', 'wardrobe'],
    'insects': ['bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'],
    'large carnivores': ['bear', 'leopard', 'lion', 'tiger', 'wolf'],
    'large man-made outdoor things': ['bridge', 'castle', 'house', 'road',
                                      'skyscraper'],
    'large natural outdoor scenes': ['cloud', 'forest', 'mountain', 'plain',
                                     'sea'],
    'large omnivores and herbivores': ['camel', 'cattle', 'chimpanzee',
                                       'elephant', 'kangaroo'],
    'medium-sized mammals': ['fox', 'porcupine', 'possum', 'raccoon', 'skunk'],
    'non-insect invertebrates': ['crab', 'lobster', 'snail', 'spider', 'worm'],
    'people': ['baby', 'boy', 'girl', 'man', 'woman'],
    'reptiles': ['crocodile', 'dinosaur', 'lizard', 'snake', 'turtle'],
    'small mammals': ['hamster', 'mouse', 'rabbit', 'shrew', 'squirrel'],
    'trees': ['maple_tree', 'oak_tree', 'palm_tree', 'pine_tree',
              'willow_tree'],
    'vehicles 1': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train'],
    'vehicles 2': ['lawn_mower', 'rocket', 'streetcar', 'tank', 'tractor'],
}

In [ ]:
if 'imagenette_builder' in locals() or 'imagenette_builder' in globals():
    del(imagenette_builder)
    print("deleting imagenette builder")
if 'datasets' in locals() or 'datasets' in globals():
    del(datasets)
    print("deleting datasets variable")
if 'training_map' in locals() or 'training_map' in globals():
    del(training_map)
    print('deleting the training map')
if 'test_map' in locals() or 'test_map' in globals():
    del(test_map)
    print('deleting the test map')
if 'training_image' in locals() or 'training_image' in globals():
    del(training_image)
    print('deleting the training image')
if 'validation_image' in locals() or 'validation_image' in globals():
    del(validation_image)
    del(training_label)
    del(validation_label)
    del(validation_examples)
    del(train_examples)
    print('deleting: validation image, training label, validation label, validation examples, train examples')

# Transformation of images

In [ ]:
for i in range(6):
    print("Processing augmentation set %i with transform %s" %(i, transform))
    x_train, y_train, x_test, y_test, my_dimension, my_iteration, nb_classes = get_data(dataname, dataname+"_augmentation"+str(i)+".dat")
    if transform == "Hilbert":
        trainstart = dt.datetime.now()
        myHilbertset = Hilbert_Transformation(copy.deepcopy(x_train), my_dimension, my_iteration)
        print("Training set transformation started at %s and finished at %s" %(trainstart.strftime('%d/%m %H:%M:%S'),dt.datetime.now().strftime('%d/%m %H:%M:%S')))
    elif transform == "Dotprint":
        # myDotprintset = dotprint_2Dto1D(copy.deepcopy(x_train))
        myHilbertset = dotprint_2Dto1D(copy.deepcopy(x_train), my_dimension, my_iteration)

    # Transform the test data:
    if transform == "Hilbert":
        teststart = dt.datetime.now()
        myHilberttest  = Hilbert_Transformation(copy.deepcopy(x_test), my_dimension, my_iteration)
        print("Test set transformation started at %s and finished at %s" %(teststart.strftime('%d%m_%H%M%S'),dt.datetime.now().strftime('%d%m_%H%M%S')))
    elif transform == "Dotprint":
        myHilberttest = dotprint_2Dto1D(copy.deepcopy(x_test), my_dimension, my_iteration)

    print(y_train.shape, y_test.shape)

    ## Setting labels to categorical variables
    if y_train.shape[-1] != nb_classes:
        y_train = np_utils.to_categorical(y_train,nb_classes) 
        y_test = np_utils.to_categorical(y_test,nb_classes)

    print(y_train.shape, y_test.shape)

    # Remove images for RAM and memory preservation
    # del(x_train)
    # del(x_test)

    # Create the dataset for each transformation
    my_datasets = {}
    my_datasets["Training"] = {}
    my_datasets["Validation"] = {}

    my_training_sets = {}
    my_training_sets["Data"] = {}

    my_validation_sets = {}
    my_validation_sets["Data"] = {}
    my_validation_sets["Label"] = {}


    my_training_sets["Data"][transform]  = myHilbertset
    ### my_training_sets["Data"]["Dotprint"] = myDotprintset
    my_training_sets["Label"] = y_train

    my_validation_sets["Data"][transform] = myHilberttest
    # # my_validation_sets["Data"]["Dotprint"] = myDotprinttest
    my_validation_sets["Label"] = y_test

    my_datasets["Training"] = my_training_sets
    my_datasets["Validation"] = my_validation_sets

    # filename = dataname+ "_" + transform + "_Aug64_Full.temp"
    exportname  = dataname+"_"+transform+'_Augmentation_'+str(i)+'.dat'
    with open(exportname, 'wb') as handle:
        pickle.dump(my_datasets, handle, protocol=pickle.HIGHEST_PROTOCOL)
        handle.close()

    del(my_datasets)
    del(myHilberttest)
    del(myHilbertset)
    del(my_training_sets)
    del(my_validation_sets)

    print("Augmentation file %i for data %s written to %s" %(i, dataname, exportname))


Processing augmentation set 0 with transform Hilbert
x_train shape: (12894, 128, 128, 3) - y_train shape: (12894, 10)
x_test shape: (500, 128, 128, 3) - y_test shape: (500, 10)
data has dimension: 2 and iteration: 7.0
Training set transformation started at 23/05 09:33:38 and finished at 23/05 09:38:56
Test set transformation started at 2305_093856 and finished at 2305_093909
(12894, 10) (500, 10)
(12894, 10) (500, 10)
Augmentation file 0 for data imagenette written to imagenette_Hilbert_Augmentation_0.dat
Processing augmentation set 1 with transform Hilbert
x_train shape: (12894, 128, 128, 3) - y_train shape: (12894, 10)
x_test shape: (500, 128, 128, 3) - y_test shape: (500, 10)
data has dimension: 2 and iteration: 7.0
Training set transformation started at 23/05 09:39:43 and finished at 23/05 09:44:55
Test set transformation started at 2305_094455 and finished at 2305_094507
(12894, 10) (500, 10)
(12894, 10) (500, 10)
Augmentation file 1 for data imagenette written to imagenette_Hilbe

In [ ]:
# print("Output written to file: %s \nin directory %s" %(filename,os.getcwd()))
print(i)

5


In [ ]:
if 0:
    setoff = np.random.randint(0,x_train.shape[0]-20)
    fig, axs=plt.subplots(4,5,figsize=(15,10))
    fig.subplots_adjust(hspace = .5, wspace=.001)

    axs = axs.ravel()

    for im_i in range(20):
        axs[im_i].set_title(class_labels[np.argmax(y_train[setoff+im_i])])
        # axs[im_i].set_title(class_labels[np.argmax(y_train[im_i])])
        axs[im_i].figsize=(15,15)
        axs[im_i].imshow(x_train[setoff+im_i])
        # axs[im_i].imshow(x_train[im_i])
        axs[im_i].axis('off')

In [ ]:
if 0:
    my_test_in = np.arange(64).reshape((8,8))
    my_test_stack = np.dstack((my_test_in, my_test_in, my_test_in))
    print(my_test_stack.shape)
    my_test_set = np.stack((my_test_stack,my_test_stack,my_test_stack,my_test_stack,my_test_stack,my_test_stack,my_test_stack,my_test_stack,my_test_stack,my_test_stack))

    my_resulting_set = Hilbert_Transformation(my_test_set,2,3)
    print(my_test_set.shape)
    print(my_resulting_set.shape)
    print(my_test_set[0])
    print(my_test_set[0].swapaxes(0,1).swapaxes(1,2)[0])
    print(my_resulting_set[0].swapaxes(0,1)[0])

In [ ]:
if 0:
    # Create the dataset for each transformation
    my_datasets = {}
    my_datasets["Training"] = {}
    my_datasets["Validation"] = {}

    my_training_sets = {}
    my_training_sets["Data"] = {}

    my_validation_sets = {}
    my_validation_sets["Data"] = {}
    my_validation_sets["Label"] = {}


    my_training_sets["Data"]["Hilbert"]  = myHilbertset
    # my_training_sets["Data"]["Dotprint"] = myDotprintset
    my_training_sets["Label"] = y_train

    my_validation_sets["Data"]["Hilbert"] = myHilberttest
    # my_validation_sets["Data"]["Dotprint"] = myDotprinttest
    my_validation_sets["Label"] = y_test

    my_datasets["Training"] = my_training_sets
    my_datasets["Validation"] = my_validation_sets

    with open(filename, 'wb') as handle:
        pickle.dump(my_datasets, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Run as script

In [ ]:
if 0:
    for transform in ['Dotprint','Hilbert']:
        for dataname in ['cifar10','cifar100','imagenette']:
            if dataname == "imagenette":
                class_labels =['tench','English springer','cassette player','chain saw','church','French horn','garbage truck','gas pump','golf ball','parachute']
                nb_classes = len(class_labels)
                imagenette_builder = tfds.builder("imagenette/full-size")
                imagenette_info = imagenette_builder.info

                imagenette_builder.download_and_prepare()
                datasets = imagenette_builder.as_dataset(as_supervised=True)

                train_size = imagenette_info.splits['train'].num_examples
                validation_size = imagenette_info.splits['validation'].num_examples

                train_examples = train_size
                validation_examples = validation_size

                im_size = 128
                training_map = datasets["train"].map(lambda image,label: (tf.image.resize(image,(im_size,im_size)),label)).take(train_examples)
                test_map = datasets["validation"].map(lambda image, label: (tf.image.resize(image, (im_size, im_size)), label)).take(validation_examples)
                rgb_dim = 3
                
                my_dimension = 2
                my_iteration = 7

                
                training_image = np.array([element[0]/256 for element in training_map])
                training_label = np.array([element[1] for element in training_map])

                validation_image = np.array([element[0]/256 for element in test_map])
                validation_label = np.array([element[1] for element in test_map])

                print("Training: %i available -> loading %i " %(train_size, train_examples))
                print("Test: %i available -> loading %i" %(validation_size, validation_examples))

                x_train = training_image.reshape(training_image.shape[0],training_image.shape[1],training_image.shape[2],training_image.shape[-1])
                x_test = validation_image.reshape(validation_image.shape[0],validation_image.shape[1],validation_image.shape[2],validation_image.shape[-1])

                # Transform the labels properly
                y_train = np_utils.to_categorical(training_label,nb_classes)
                y_test = np_utils.to_categorical(validation_label,nb_classes)
            elif dataname == 'cifar10':
                class_labels = ["airplane","car","bird","cat","deer","dog","frog","horse","ship","truck"]
                nb_classes = len(class_labels)
                
                my_dimension = 2
                my_iteration = 5

                (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
                y_train = np_utils.to_categorical(y_train,nb_classes)
                y_test = np_utils.to_categorical(y_test,nb_classes)

                x_train = np.array([element/256 for element in x_train])
                x_test  = np.array([element/256 for element in x_test])

                input_shape = x_train[0].shape
                print(f"x_train shape: {x_train.shape} - y_train shape: {y_train.shape}")
                print(f"x_test shape: {x_test.shape} - y_test shape: {y_test.shape}")
            elif dataname == 'cifar100':
                nb_classes = 100
                my_dimension = 2
                my_iteration = 5

                (x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data()
                y_train = np_utils.to_categorical(y_train,nb_classes)
                y_test = np_utils.to_categorical(y_test,nb_classes)

                x_train = np.array([element/256 for element in x_train])
                x_test  = np.array([element/256 for element in x_test])
                
                input_shape = x_train[0].shape

                print(f"x_train shape: {x_train.shape} - y_train shape: {y_train.shape}")
                print(f"x_test shape: {x_test.shape} - y_test shape: {y_test.shape}")

            filename  = dataname+"_"+transform+'.dat'
            print("Dataset output written to %s" %filename)


            if dataname == 'cifar100':
                class_labels = [
                'apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle','bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel','can','castle','caterpillar','cattle','chair','chimpanzee','clock','cloud','cockroach','couch','crab','crocodile','cup',
                'dinosaur','dolphin','elephant','flatfish','forest','fox','girl','hamster','house','kangaroo','computer_keyboard','lamp','lawn_mower','leopard','lion','lizard','lobster','man','maple_tree','motorcycle','mountain','mouse','mushroom',
                'oak_tree','orange','orchid','otter','palm_tree','pear','pickup_truck','pine_tree','plain','plate','poppy','porcupine','possum','rabbit','raccoon','ray','road','rocket','rose','sea','seal','shark','shrew','skunk','skyscraper','snail','snake',
                'spider','squirrel','streetcar','sunflower','sweet_pepper','table','tank','telephone','television','tiger','tractor','train','trout','tulip','turtle','wardrobe','whale','willow_tree','wolf','woman','worm',
            ]

            mapping_coarse_fine = {
                'aquatic mammals': ['beaver', 'dolphin', 'otter', 'seal', 'whale'],
                'fish': ['aquarium_fish', 'flatfish', 'ray', 'shark', 'trout'],
                'flowers': ['orchid', 'poppy', 'rose', 'sunflower', 'tulip'],
                'food containers': ['bottle', 'bowl', 'can', 'cup', 'plate'],
                'fruit and vegetables': ['apple', 'mushroom', 'orange', 'pear',
                                        'sweet_pepper'],
                'household electrical device': ['clock', 'computer_keyboard', 'lamp',
                                                'telephone', 'television'],
                'household furniture': ['bed', 'chair', 'couch', 'table', 'wardrobe'],
                'insects': ['bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'],
                'large carnivores': ['bear', 'leopard', 'lion', 'tiger', 'wolf'],
                'large man-made outdoor things': ['bridge', 'castle', 'house', 'road',
                                                'skyscraper'],
                'large natural outdoor scenes': ['cloud', 'forest', 'mountain', 'plain',
                                                'sea'],
                'large omnivores and herbivores': ['camel', 'cattle', 'chimpanzee',
                                                'elephant', 'kangaroo'],
                'medium-sized mammals': ['fox', 'porcupine', 'possum', 'raccoon', 'skunk'],
                'non-insect invertebrates': ['crab', 'lobster', 'snail', 'spider', 'worm'],
                'people': ['baby', 'boy', 'girl', 'man', 'woman'],
                'reptiles': ['crocodile', 'dinosaur', 'lizard', 'snake', 'turtle'],
                'small mammals': ['hamster', 'mouse', 'rabbit', 'shrew', 'squirrel'],
                'trees': ['maple_tree', 'oak_tree', 'palm_tree', 'pine_tree',
                        'willow_tree'],
                'vehicles 1': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train'],
                'vehicles 2': ['lawn_mower', 'rocket', 'streetcar', 'tank', 'tractor'],
            }

            if 'imagenette_builder' in locals() or 'imagenette_builder' in globals():
                del(imagenette_builder)
            if 'datasets' in locals() or 'datasets' in globals():
                del(datasets)
            if 'training_map' in locals() or 'training_map' in globals():
                del(training_map)
            if 'test_map' in locals() or 'test_map' in globals():
                del(test_map)
            if 'training_image' in locals() or 'training_image' in globals():
                del(training_image)
            if 'validation_image' in locals() or 'validation_image' in globals():
                del(validation_image)
                del(training_label)
                del(validation_label)
                del(validation_examples)
                del(train_examples)
                
            # Transform the datasets to linearized versions
            if transform == "Hilbert":
                # myHilbertset  = Hilbert_collection(copy.deepcopy(x_train),my_dimension,my_iteration) 
                myHilbertset  = Hilbert_collection(x_train,my_dimension,my_iteration) 
            elif transform == "Dotprint":
                # myDotprintset = dotprint_2Dto1D(copy.deepcopy(x_train))
                myHilbertset = dotprint_2Dto1D(x_train)

            # Transform the test data:
            if transform == "Hilbert":
                myHilberttest  = Hilbert_collection(copy.deepcopy(x_test),my_dimension, my_iteration)
            elif transform == "Dotprint":
                myHilberttest = dotprint_2Dto1D(copy.deepcopy(x_test))

            # Remove images for RAM and memory preservation
            del(x_train)
            del(x_test)

            # Create the dataset for each transformation
            my_datasets = {}
            my_datasets["Training"] = {}
            my_datasets["Validation"] = {}

            my_training_sets = {}
            my_training_sets["Data"] = {}

            my_validation_sets = {}
            my_validation_sets["Data"] = {}

            my_training_sets["Data"][transform]  = myHilbertset
            my_training_sets["Label"] = y_train

            my_validation_sets["Data"][transform] = myHilberttest
            my_validation_sets["Label"] = y_test

            my_datasets["Training"] = my_training_sets
            my_datasets["Validation"] = my_validation_sets

            with open(filename, 'wb') as handle:
                pickle.dump(my_datasets, handle, protocol=pickle.HIGHEST_PROTOCOL)
                handle.close()

            del(my_datasets)
            del(myHilberttest)
            del(myHilbertset)
            del(my_training_sets)
            del(my_validation_sets)